In [17]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.layers import Layer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils import shuffle

In [18]:
CSV_PATH = "training_data_final.csv"
CATALOG_PATH = "habit_catalog.json"

MODEL_DIR = "deepfm_model"
os.makedirs(MODEL_DIR, exist_ok=True)

PREPROC_PATH = os.path.join(MODEL_DIR, "preproc.joblib")
MODEL_PATH = os.path.join(MODEL_DIR, "model.keras")


In [19]:
# 1. LOAD DATA
# -------------------------------------------------------
df = pd.read_csv(CSV_PATH)
df["label"] = df["label"].astype(int)


In [20]:
numeric_cols = [
    "risk_score",
    "prediction",
    "emotion_score",
    "screen_time",
    "unlocks",
    "sleep_hours",
    "steps_last_24h",
    "popularity_prior"
]


categorical_cols = [
    "habit_id",
    "category",
    "dopamine_level",
    "difficulty",
    "time_min",
    "indoor",
    "required_device",
    "dominant_emotion",
    "recommended_for_state"
]


# ----------------------------------------------
# CLEAN recommended_for_state (CRITICAL)
# ----------------------------------------------
def normalize_state(v):
    if isinstance(v, list):
        return v[0] if len(v) else "unknown"
    if isinstance(v, str):
        if v.startswith("[") and "," in v:
            parts = v.strip("[] ").split(",")
            return parts[0].replace("'", "").strip()
        return v
    return "unknown"

df["recommended_for_state"] = df["recommended_for_state"].apply(normalize_state)



In [6]:
# ----------------------------------------------
# CLEAN missing values
# ----------------------------------------------
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

for c in categorical_cols:
    df[c] = df[c].astype(str).fillna("unknown")

In [22]:
# ----------------------------------------------
# Encode categoricals
# ----------------------------------------------
encoders = {}

for c in categorical_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])
    encoders[c] = le

# ----------------------------------------------
# Scale numeric
# ----------------------------------------------
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


# save
preproc = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "encoders": encoders,
    "scaler": scaler
}

joblib.dump(preproc, PREPROC_PATH)


['deepfm_model\\preproc.joblib']

In [23]:
X_cat = {c: df[c].values for c in categorical_cols}
X_num = df[numeric_cols].values
y = df["label"].values

train_idx, val_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.2,
    random_state=42,
    stratify=y
)

def build_inputs(idx):
    out = {}
    for c in categorical_cols:
        out[c] = X_cat[c][idx]
    out["numeric_input"] = X_num[idx]
    return out

train_in = build_inputs(train_idx)
val_in = build_inputs(val_idx)

def keras_inputs(inp):
    out = {}
    for c in categorical_cols:
        out[f"inp_{c}"] = inp[c].astype("int32")
    out["numeric_input"] = inp["numeric_input"].astype("float32")
    return out

train_keras = keras_inputs(train_in)
val_keras = keras_inputs(val_in)

y_train = y[train_idx]
y_val = y[val_idx]


In [24]:
class FMLayer(Layer):
    def call(self, inputs):
        summed = tf.reduce_sum(inputs, axis=1)
        summed_square = tf.square(summed)

        squared = tf.square(inputs)
        squared_sum = tf.reduce_sum(squared, axis=1)

        return 0.5 * tf.reduce_sum(summed_square - squared_sum, axis=1, keepdims=True)


In [25]:
EMB_DIM = 8
inputs = {}
emb_fm_list = []
emb_dnn_list = []

for c in categorical_cols:
    vocab = df[c].nunique() + 1
    inp = Input(shape=(1,), name=f"inp_{c}")
    inputs[c] = inp

    emb = layers.Embedding(vocab, EMB_DIM)(inp)
    emb_fm_list.append(emb)
    emb_dnn_list.append(layers.Reshape((EMB_DIM,))(emb))

# numeric
num_inp = Input(shape=(len(numeric_cols),), name="numeric_input")
inputs["numeric_input"] = num_inp

num_fm = layers.Dense(EMB_DIM)(num_inp)
num_fm = layers.Reshape((1, EMB_DIM))(num_fm)
emb_fm_list.append(num_fm)

emb_dnn_list.append(num_inp)

# FM
fm_tensor = layers.Concatenate(axis=1)(emb_fm_list)
fm_out = FMLayer()(fm_tensor)

# Linear
linear_logit = layers.Dense(1)(layers.Concatenate()(emb_dnn_list))

# DNN
dnn_input = layers.Concatenate()(emb_dnn_list)
x = layers.Dense(128, activation="relu")(dnn_input)
x = layers.Dropout(0.2)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.2)(x)
dnn_logit = layers.Dense(1)(x)

# Final
final_logit = layers.Add()([fm_out, linear_logit, dnn_logit])
output = layers.Activation("sigmoid")(final_logit)

model = Model(inputs=list(inputs.values()), outputs=output)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.AUC(name="auc")]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp_habit_id        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_category        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_dopamine_level  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_difficulty      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_time_min        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_indoor          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_required_device │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_dominant_emoti… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_recommended_fo… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 8)      │      2,408 │ inp_habit_id[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 8)      │         32 │ inp_category[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 8)      │         32 │ inp_dopamine_lev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 8)      │         48 │ inp_difficulty[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 8)      │         96 │ inp_time_min[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 8)      │         24 │ inp_indoor[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 1, 8)      │         32 │ inp_required_dev… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 1, 8)      │        232 │ inp_dominant_emo

 Total params: 21,906 (85.57 KB)

 Trainable params: 21,906 (85.57 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", patience=5, mode="max", restore_best_weights=True
    )
]

model.fit(
    train_keras, y_train,
    validation_data=(val_keras, y_val),
    epochs=25,
    batch_size=512,
    callbacks=callbacks,
    verbose=2
)


Epoch 1/25
32/32 - 3s - 107ms/step - auc: 0.6435 - loss: 0.6198 - val_auc: 0.7345 - val_loss: 0.5645
Epoch 2/25
32/32 - 0s - 9ms/step - auc: 0.7333 - loss: 0.5649 - val_auc: 0.7439 - val_loss: 0.5573
Epoch 3/25
32/32 - 0s - 9ms/step - auc: 0.7475 - loss: 0.5523 - val_auc: 0.7493 - val_loss: 0.5533
Epoch 4/25
32/32 - 0s - 8ms/step - auc: 0.7595 - loss: 0.5436 - val_auc: 0.7521 - val_loss: 0.5521
Epoch 5/25
32/32 - 0s - 7ms/step - auc: 0.7673 - loss: 0.5372 - val_auc: 0.7529 - val_loss: 0.5518
Epoch 6/25
32/32 - 0s - 6ms/step - auc: 0.7730 - loss: 0.5317 - val_auc: 0.7545 - val_loss: 0.5516
Epoch 7/25
32/32 - 0s - 7ms/step - auc: 0.7783 - loss: 0.5269 - val_auc: 0.7533 - val_loss: 0.5527
Epoch 8/25
32/32 - 0s - 6ms/step - auc: 0.7820 - loss: 0.5237 - val_auc: 0.7535 - val_loss: 0.5536
Epoch 9/25
32/32 - 0s - 11ms/step - auc: 0.7845 - loss: 0.5212 - val_auc: 0.7526 - val_loss: 0.5561
Epoch 10/25
32/32 - 0s - 6ms/step - auc: 0.7897 - loss: 0.5162 - val_auc: 0.7517 - val_loss: 0.5563
Epoch 

In [48]:
# 9. Evaluate
# -------------------------------------------------------
val_pred = model.predict(val_keras_inputs).ravel()
val_bin = (val_pred >= 0.5).astype(int)

print("\n----- Validation Report -----")
print(classification_report(y_val, val_bin))
print("ROC AUC:", roc_auc_score(y_val, val_pred))


125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

----- Validation Report -----
              precision    recall  f1-score   support

           0       0.63      0.44      0.52      1396
           1       0.74      0.86      0.80      2604

    accuracy                           0.71      4000
   macro avg       0.69      0.65      0.66      4000
weighted avg       0.70      0.71      0.70      4000

ROC AUC: 0.7487277122698603


In [49]:
# -------------------------------------------------------
# 10. Save final model (Keras 3 compatible)
# -------------------------------------------------------

model.save(MODEL_PATH)
print("Model saved to:", MODEL_PATH)
print("Preprocessing saved to:", PREPROC_PATH)


Model saved to: deepfm_model\model.keras
Preprocessing saved to: deepfm_model\preproc.joblib


## Inference

In [50]:
def load_all():
    pre = joblib.load(PREPROC_PATH)
    mdl = tf.keras.models.load_model(MODEL_PATH, custom_objects={"FMLayer": FMLayer})
    return pre, mdl


In [51]:
def normalize_state_inf(v):
    if isinstance(v, list):
        return v[0] if len(v) else "unknown"
    return str(v)

def build_candidates(user_row, catalog):
    df_user = pd.DataFrame([user_row])
    cat = catalog.copy()
    cat["recommended_for_state"] = cat["recommended_for_state"].apply(normalize_state_inf)
    df_user_rep = pd.concat([df_user] * len(cat), ignore_index=True)
    return pd.concat([df_user_rep, cat], axis=1)



In [52]:
def preprocess_inf(df, pre):
    encs = pre["encoders"]
    scaler = pre["scaler"]
    cat_cols = pre["categorical_cols"]
    num_cols = pre["numeric_cols"]

    df = df.copy()

    for c in cat_cols:
        df[c] = df[c].astype(str)
        le = encs[c]
        df[c] = df[c].apply(lambda x: le.transform([x])[0] if x in le.classes_ else 0)

    df[num_cols] = scaler.transform(df[num_cols])

    inp = {}
    for c in cat_cols:
        inp[f"inp_{c}"] = df[c].values.astype("int32")
    inp["numeric_input"] = df[num_cols].values.astype("float32")

    return inp, df


In [53]:
def recommend(user_row, catalog, top_k=5):
    pre, mdl = load_all()
    df_cand = build_candidates(user_row, catalog)
    inp, df_proc = preprocess_inf(df_cand, pre)

    scores = mdl.predict(inp, batch_size=512).ravel()
    df_proc["score"] = scores

    df_sorted = df_proc.sort_values("score", ascending=False)
    return df_sorted.head(top_k)[["habit_id", "score"]]


In [ ]:
# user snapshot as dict (names must match preproc)
user_snapshot_1 = {
    "risk_score": 0.82,             # high risk
    "prediction": 1,                # predicted low performance/mood
    "dominant_emotion": "Sadness",   # emotional state
    "emotion_score": 0.83,          # strong negative emotion
    "screen_time": 360,             # high screen time (minutes)
    "unlocks": 75,                  # high phone usage
    "sleep_hours": 9.2,             # high sleep
    "steps_last_24h": 1800          # low activity
}

user_snapshot_2 = {
    "risk_score": 0.18,               # low risk
    "prediction": 0,                  # stable predicted mood/performance
    "dominant_emotion": "Joy",       # positive emotion
    "emotion_score": -0.61,           # low emotional activation
    "screen_time": 195,                # healthy usage
    "unlocks": 32,                    # low distractibility
    "sleep_hours": 7.6,               # good sleep
    "steps_last_24h": 6200            # active user
}



# habit_catalog: load your CSV or JSON into df_habits
df_habits = pd.read_json("habit_catalog.json")# or your DataFrame

pos5 = recommend(user_snapshot_1, df_habits, top_k=5)
pos6 = recommend(user_snapshot_2, df_habits, top_k=5)
print("Top 5 recs for user 1:\n", pos5)
print()
print("Top 5 recs for user 2:\n", pos6)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 245ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
Top 5 recs for user 1:
      habit_id     score
54          0  0.996582
110         0  0.995055
146         0  0.994891
25          0  0.994891
101         0  0.994891

Top 5 recs for user 2:
      habit_id     score
114         0  0.955948
54          0  0.948304
184         0  0.932062
6           0  0.930561
32          0  0.929543
